# Chapter 7: ML Pipelines, Feature Stores, and MLOps

This notebook complements Chapter 7 by turning release discipline into scenario comparisons.

The chapter now distinguishes several release artifacts explicitly:
- model registry
- feature registry
- prompt registry
- corpus registry
- evaluation registry
- deployment manifest

Use the retail ranking track to review feature reuse, rollout discipline, and rollback authority. Use the support-assistant track to widen that same release contract into LLMOps.

In [ ]:
from pathlib import Path

def find_companion_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not find companion repository root.")


def ensure_path_exists(path: Path, kind: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected {kind}: {path}. "
            "Make sure you are running this notebook from inside the companion repository."
        )
    return path

NOTEBOOK_DIR = Path.cwd()
COMPANION_ROOT = find_companion_root(NOTEBOOK_DIR)
print("Companion root located successfully.")

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location

import pandas as pd

module_path = ensure_path_exists(
    COMPANION_ROOT / "code" / "integration" / "ml_integration_planning.py",
    "helper module",
)
spec = spec_from_file_location("ml_integration_planning", module_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load helper module spec from {module_path}")
ml_integration_planning = module_from_spec(spec)
spec.loader.exec_module(ml_integration_planning)

load_integration_scenarios = ml_integration_planning.load_integration_scenarios
summarize_integration_pressure = ml_integration_planning.summarize_integration_pressure
build_integration_plan = ml_integration_planning.build_integration_plan
find_fragile_integrations = ml_integration_planning.find_fragile_integrations

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)

### Why the Setup Code Matters

The helper module is modeling release discipline, not just model metadata. That is why the inputs include evaluation discipline, owner clarity, rollback readiness, and serving criticality in addition to feature reuse and registry state. The chapter's point is that reproducibility and deployability are related but not identical.

## Part 1: Load Integration Scenarios

The scenario table now includes evaluation discipline, benchmark versioning, and owner clarity in addition to feature reuse, latency, and registry state.


In [ ]:
scenario_path = ensure_path_exists(
    COMPANION_ROOT / "datasets" / "integration" / "sample_mlops_scenarios.csv",
    "dataset",
)
scenarios_df = load_integration_scenarios(scenario_path)
scenarios_df[[
    "scenario_name",
    "model_type",
    "registry_discipline",
    "evaluation_discipline",
    "benchmark_versioned",
    "owner_clarity",
]].sort_values("scenario_name").reset_index(drop=True)

### What to Notice in the Scenario Table

The scenario table should make release discipline visible before any model is deployed. Pay attention to registry quality, benchmark versioning, rollback readiness, and owner clarity, because those are often the fields that explain later rollout failures.


## Part 2: Summarize Integration Pressure

This summary now shows where weak evaluation discipline or unclear ownership boundaries are starting to matter as much as latency and feature reuse.


In [ ]:
summary_df = summarize_integration_pressure(scenarios_df)
summary_df


### Interpreting the Integration Summary

The summary shows where the platform is brittle at the portfolio level. If weak evaluation discipline or unclear ownership appears repeatedly, the organization has a release-process problem, not just a single bad model run.


## Part 3: Build an Integration Plan

The helper now attaches evaluation and ownership contracts in addition to feature, deployment, and feedback recommendations.


In [ ]:
plan_df = build_integration_plan(scenarios_df)
plan_df[[
    "scenario_name",
    "feature_contract",
    "deployment_pattern",
    "evaluation_contract",
    "ownership_contract",
    "primary_risk",
    "recommended_action",
]].sort_values("scenario_name").reset_index(drop=True)


### Interpreting the Integration Plan

Read the plan row by row as a deployment contract. The useful question is whether feature handling, evaluation lineage, ownership, and recommended action all align well enough that a release team could actually act on them.


## Part 4: Review Deployment Workflow and Rollout Readiness

Promotion should not jump straight from approval to full exposure. Use this view to reason through candidate review, shadowing, canary release, limited rollout, monitoring signals, and rollback triggers. In LLM-centered systems, the same workflow also has to preserve prompt, retrieval, and tool state.

In [ ]:
rollout_view = plan_df[[
    "scenario_name",
    "serving_criticality",
    "deployment_pattern",
    "rollback_ready",
    "owner_clarity",
    "recommended_action",
]].sort_values(["serving_criticality", "scenario_name"], ascending=[False, True]).reset_index(drop=True)
rollout_view


### What the Rollout View Should Tell You

This output should tell you whether a scenario is ready for shadowing, canary rollout, or an immediate pause. If rollback readiness is weak while serving criticality is high, the safest interpretation is that deployment mechanics are outrunning release discipline.


## Part 5: Inspect Evaluation Pipelines and Benchmark Lineage

Reliable promotion depends on more than model artifacts. It also depends on versioned benchmarks, stable evaluation rules, and enough lineage to separate model improvement from evaluation drift.

In [ ]:
evaluation_view = plan_df[[
    "scenario_name",
    "model_type",
    "evaluation_discipline",
    "benchmark_versioned",
    "evaluation_contract",
    "primary_risk",
]].sort_values(["evaluation_discipline", "scenario_name"]).reset_index(drop=True)
evaluation_view


### What the Evaluation View Should Tell You

Use this table to separate model quality from comparison quality. A scenario can have promising model work and still be unfit for promotion if the benchmark lineage or evaluation contract is weak.


## Part 6: Inspect Ownership Boundaries and Platform Responsibilities

Many integration failures come from unclear ownership rather than missing software. Extend the view beyond feature freshness and benchmark integrity: who owns the system prompt, few-shot examples, retrieval policy, tool schema, and the final release decision when those signals disagree?

In [ ]:
ownership_view = plan_df[[
    "scenario_name",
    "serving_criticality",
    "owner_clarity",
    "ownership_contract",
    "primary_risk",
    "recommended_action",
]].sort_values(["owner_clarity", "scenario_name"]).reset_index(drop=True)
ownership_view


### What the Ownership View Should Tell You

The ownership view is really an incident-response preview. If the responsible party is unclear before deployment, the platform will discover that ambiguity at the worst possible moment, usually during rollback or degraded serving behavior.


## Part 7: Inspect Fragile Integrations

These are the scenarios most likely to fail through skew, weak promotion discipline, evaluation drift, or unclear ownership during rollback.


In [ ]:
fragile_df = find_fragile_integrations(plan_df)
fragile_df


### How to Read the Integration Recommendations

Use the fragile-scenario output as a release-discipline check. Each row is telling you which contract is likely to fail first: feature alignment, benchmark lineage, rollout readiness, ownership clarity, or rollback authority. That is the bridge from reproducible training to trustworthy deployment.


## Part 7A: Inspect Release Artifacts

The chapter names explicit release artifacts because deployment discipline should survive outside the scenario table. The snippets below show how the companion captures feature, promotion, and release-readiness state in lightweight files.

In [ ]:
integration_artifact_dir = ensure_path_exists(
    COMPANION_ROOT / "artifacts" / "integration",
    "artifact directory",
)
print("feature_registry.yaml")
print("-" * 60)
print("\n".join((ensure_path_exists(integration_artifact_dir / "feature_registry.yaml", "artifact")).read_text(encoding="utf-8").splitlines()[:18]))
print("\nmodel_promotion_record.yaml")
print("-" * 60)
print("\n".join((ensure_path_exists(integration_artifact_dir / "model_promotion_record.yaml", "artifact")).read_text(encoding="utf-8").splitlines()[:18]))
print("\nrelease_readiness.yaml")
print("-" * 60)
print("\n".join((ensure_path_exists(integration_artifact_dir / "release_readiness.yaml", "artifact")).read_text(encoding="utf-8").splitlines()[:18]))
print("\nml_integration_planning.py")
print("-" * 60)
print("\n".join(module_path.read_text(encoding="utf-8").splitlines()[:18]))

## Part 7B: Compare Registry Responsibilities

Release discipline gets easier to reason about when the different registries are shown side by side.

In [ ]:
registry_comparison = pd.DataFrame([
    {'registry': 'model registry', 'primary_question': 'Which learned artifact was approved?', 'failure_if_missing': 'model lineage is weak at promotion time'},
    {'registry': 'feature registry', 'primary_question': 'Which feature definitions were expected online?', 'failure_if_missing': 'training-serving skew hides inside local feature logic'},
    {'registry': 'prompt registry', 'primary_question': 'Which prompt or system instruction was live?', 'failure_if_missing': 'behavior changes without a clear release record'},
    {'registry': 'corpus registry', 'primary_question': 'Which corpus snapshot and chunking policy were live?', 'failure_if_missing': 'retrieval quality changes without explainable source state'},
    {'registry': 'evaluation registry', 'primary_question': 'Which benchmark and rubric approved the release?', 'failure_if_missing': 'comparison quality drifts without being noticed'},
])
registry_comparison

## Part 7C: Generate a Deployment Manifest

The chapter treats the deployment manifest as the central release artifact because it binds the model, data, prompt, retrieval, tool, and rollback state into one record.

In [ ]:
def build_deployment_manifest(scenario_name):
    row = plan_df.loc[plan_df['scenario_name'] == scenario_name].iloc[0]
    return {
        'application': scenario_name,
        'model_version': 'llm_support_v4' if 'rag' in scenario_name else 'ranking_model_v12',
        'feature_dependencies': 'session_features_v8' if row['model_type'] != 'retrieval_augmented_llm' else 'customer_profile_features_v3',
        'prompt_template_version': 'support_answer_template_v7' if 'rag' in scenario_name else 'n/a',
        'corpus_snapshot': 'support_docs_v14' if 'rag' in scenario_name else 'n/a',
        'vector_index_version': 'support_index_v14_2' if 'rag' in scenario_name else 'n/a',
        'tool_versions': 'order_status_lookup_v2' if 'rag' in scenario_name else 'n/a',
        'benchmark_version': 'support_eval_v9' if 'rag' in scenario_name else 'ranking_benchmark_v5',
        'rollout_pattern': row['deployment_pattern'],
        'rollback_target': {
            'prompt_template_version': 'support_answer_template_v6',
            'corpus_version': 'support_docs_v13',
            'retrieval_index_version': 'support_index_v13_4',
            'tool_version': 'order_status_lookup_v1',
        } if 'rag' in scenario_name else 'ranking_model_v11 + session_features_v7',
        'release_owner': row['ownership_contract'],
    }

manifest = build_deployment_manifest('rag_support_assistant')
pd.Series(manifest)

## Part 7D: Canary Signals and Release Gate

A release gate is useful only if the notebook turns monitoring signals into a go, canary, pause, or rollback decision.

In [ ]:
canary_signals = pd.DataFrame([
    {'signal': 'retrieval_groundedness', 'severity': 'critical', 'status': 'fail', 'note': 'unauthorized-context test failed'},
    {'signal': 'latency_budget', 'severity': 'moderate', 'status': 'pass', 'note': 'response time stayed inside target'},
    {'signal': 'tool_schema_validation', 'severity': 'high', 'status': 'pass', 'note': 'tool outputs matched approved schema'},
])

def release_gate(signals):
    if ((signals['severity'] == 'critical') & (signals['status'] == 'fail')).any():
        return 'pause'
    if ((signals['severity'] == 'high') & (signals['status'] == 'fail')).any():
        return 'canary only'
    if (signals['status'] == 'fail').any():
        return 'proceed with warning'
    return 'proceed'

release_decision = release_gate(canary_signals)
print(f'release_gate = {release_decision}')
canary_signals

## Part 7E: Compare a Clean Release Case

A failing canary is useful, but the notebook should also show what a release-ready signal set looks like when the gates are satisfied.

In [ ]:
clean_release_signals = pd.DataFrame([
    {'signal': 'retrieval_groundedness', 'severity': 'critical', 'status': 'pass', 'note': 'groundedness benchmark stayed above threshold'},
    {'signal': 'latency_budget', 'severity': 'moderate', 'status': 'pass', 'note': 'response time stayed inside target'},
    {'signal': 'tool_schema_validation', 'severity': 'high', 'status': 'pass', 'note': 'tool outputs matched approved schema'},
])
clean_release_decision = release_gate(clean_release_signals)
print(f'clean_release_gate = {clean_release_decision}')
clean_release_signals

## Part 7F: Why Model Rollback Alone Can Fail

The release lab should show why MLOps becomes LLMOps when prompt, corpus, and tool state are also live dependencies.

In [ ]:
rollback_gap = pd.DataFrame([
    {'artifact': 'model_version', 'rollback_status': 'rolled back', 'still_live_problem': 'none'},
    {'artifact': 'prompt_template_version', 'rollback_status': 'unchanged', 'still_live_problem': 'unsafe refusal behavior is still live'},
    {'artifact': 'vector_index_version', 'rollback_status': 'unchanged', 'still_live_problem': 'assistant still retrieves outdated or unauthorized context'},
    {'artifact': 'tool_schema_version', 'rollback_status': 'unchanged', 'still_live_problem': 'tool-call formatting risk remains in production'},
])
rollback_gap

## Part 8: Prompt Regression Release Gate

Chapter 7 should also end with one compact prompt-regression release gate. The fuller cross-chapter lab lives in `prompt_regression_release_lab.ipynb`, but this smaller section keeps the core release-decision logic inside the Chapter 7 notebook.

In [ ]:
prompt_release_cases = pd.DataFrame([
    {'case_id': 'support_001', 'prompt_version': 'support_answer_template_v7', 'corpus_version': 'support_docs_v14', 'retrieval_index_version': 'support_index_v14_2', 'failure_category': 'hallucinated_policy', 'severity': 'critical', 'pass_fail': 'fail'},
    {'case_id': 'support_002', 'prompt_version': 'support_answer_template_v7', 'corpus_version': 'support_docs_v14', 'retrieval_index_version': 'support_index_v14_2', 'failure_category': 'unsafe_tool_use', 'severity': 'moderate', 'pass_fail': 'pass'},
    {'case_id': 'support_003', 'prompt_version': 'support_answer_template_v7', 'corpus_version': 'support_docs_v14', 'retrieval_index_version': 'support_index_v14_2', 'failure_category': 'stale_context', 'severity': 'high', 'pass_fail': 'fail'},
    {'case_id': 'support_004', 'prompt_version': 'support_answer_template_v7', 'corpus_version': 'support_docs_v14', 'retrieval_index_version': 'support_index_v14_2', 'failure_category': 'unauthorized_context', 'severity': 'critical', 'pass_fail': 'fail'},
])
critical_prompt_failures = int(((prompt_release_cases['severity'] == 'critical') & (prompt_release_cases['pass_fail'] == 'fail')).sum())
high_prompt_failures = int(((prompt_release_cases['severity'] == 'high') & (prompt_release_cases['pass_fail'] == 'fail')).sum())
moderate_prompt_failures = int(((prompt_release_cases['severity'] == 'moderate') & (prompt_release_cases['pass_fail'] == 'fail')).sum())
if critical_prompt_failures > 0:
    prompt_release_outcome = 'pause'
elif high_prompt_failures > 0:
    prompt_release_outcome = 'canary only'
elif moderate_prompt_failures > 0:
    prompt_release_outcome = 'proceed with warning'
else:
    prompt_release_outcome = 'proceed'
prompt_rollback_target = {
    'prompt_template_version': 'support_answer_template_v6',
    'corpus_version': 'support_docs_v13',
    'retrieval_index_version': 'support_index_v13_4',
    'tool_version': 'order_status_lookup_v1',
}
print(f'critical_prompt_failures = {critical_prompt_failures}')
print(f'high_prompt_failures = {high_prompt_failures}')
print(f'moderate_prompt_failures = {moderate_prompt_failures}')
print(f'prompt_release_outcome = {prompt_release_outcome}')
print(f'prompt_rollback_target = {prompt_rollback_target}')
prompt_release_cases

## Exercise

### Concept check
- Compare a retail-ranking release with a support-assistant release and name which manifest fields are shared versus which appear only once prompt, corpus, embedding, vector, or tool state become operational.

### Diagnostic scenario
- Use the failing canary signals to decide whether release should proceed, warn, stay canary-only, pause, or roll back. Then explain why bad behavior can continue even after model rollback.

### Artifact-building exercise
- Build two deployment manifests: one for a traditional ML release and one for a support-assistant release that adds prompt template, corpus snapshot, chunking configuration, embedding model, vector index, and tool schema.

### Notebook extension
- Compare the clean release case with the blocked release case and state which owner has authority to pause or reverse the release, which manifest field matters most for diagnosis, and why model rollback alone is not enough.
- Use `prompt_regression_release_lab.ipynb` to compare the release gate, rollback manifest, and incident reconstruction bundle for the same assistant release.